# HVAC Energy Optimization — Physics-Informed ML Study
Author: Sabbir Hossain

1. Generate synthetic commercial-building data
2. Train a cooling-load forecaster (GBM)
3. Compare baseline vs SAT-reset vs chiller sequencing
4. Evaluate with ASHRAE Guideline 14 metrics

In [ ]:
from hvac_optimizer.synthetic import generate_year
df = generate_year(n_days=90)
df.head(), df.describe()

In [ ]:
from hvac_optimizer.forecasting import add_time_features, add_lag_features, CoolingLoadForecaster
import pandas as pd
d = add_lag_features(add_time_features(df), 'cooling_load_kw').dropna()
feats = [c for c in d.columns if c != 'cooling_load_kw' and pd.api.types.is_numeric_dtype(d[c])]
s = int(len(d)*0.8)
fx = CoolingLoadForecaster(model='gbm').fit(d[feats].iloc[:s], d['cooling_load_kw'].iloc[:s])
fx.evaluate(d[feats].iloc[s:], d['cooling_load_kw'].iloc[s:])

In [ ]:
fx.importances().head(10)

In [ ]:
from hvac_optimizer.optimization import baseline_control, optimize_supply_air_temp, optimize_chiller_sequencing
from hvac_optimizer.evaluation import energy_savings_pct
test = d.iloc[s:]
b = baseline_control(test['cooling_load_kw'], test['t_out'])
o = optimize_supply_air_temp(test['cooling_load_kw'], test['t_out'])
q = optimize_chiller_sequencing(test['cooling_load_kw'], test['t_out'])
eb, eo, eq = b['p_elec_kw'].sum(), o['p_elec_kw'].sum(), q['p_elec_kw'].sum()
print(f'baseline={eb:.0f} kWh  SAT-reset={eo:.0f} kWh ({energy_savings_pct(eb,eo):.1f}%)  sequencing={eq:.0f} kWh ({energy_savings_pct(eb,eq):.1f}%)')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2,1,figsize=(10,6),sharex=True)
ax[0].plot(test['cooling_load_kw'].iloc[:168].values, label='load kW')
ax[0].legend()
ax[1].plot(b['p_elec_kw'].iloc[:168].values, label='baseline')
ax[1].plot(o['p_elec_kw'].iloc[:168].values, label='SAT-reset')
ax[1].legend(); plt.tight_layout(); plt.show()